In [1]:
import pandas as pd
import numpy as np
import os
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# -----------------------------
# Prepare spectral features
# -----------------------------
spectral_cols = [c for c in train.columns if c not in ["sample number", "species number", "樹種", "含水率"]]

X = train[spectral_cols].values
y = train["含水率"].values
X_test = test[spectral_cols].values

print("Number of spectral features:", X.shape[1])

# -----------------------------
# Savitzky-Golay smoothing + 1st derivative
# -----------------------------
# window_length and polyorder chosen based on typical NIR practice
X_sg = savgol_filter(X, window_length=11, polyorder=2, deriv=1, axis=1)
X_test_sg = savgol_filter(X_test, window_length=11, polyorder=2, deriv=1, axis=1)

# -----------------------------
# Scale features
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sg)
X_test_scaled = scaler.transform(X_test_sg)

print("Preprocessing complete: smoothing + derivative + scaling")

# -----------------------------
# Train ElasticNet model
# -----------------------------
# Using standard alpha and l1_ratio; matches the simplicity of the paper's regression
model = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=100000, tol=1e-3, random_state=42)
model.fit(X_scaled, y)

preds = model.predict(X_test_scaled)
print("Sample predictions:", preds[:10])

# -----------------------------
# Save submission
# -----------------------------
os.makedirs("../submissions", exist_ok=True)
submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": preds
})

output_path = "../submissions/exp_nir_article_style.csv"
submission.to_csv(output_path, index=False, header=False)

print("\nSubmission saved to:", output_path)
check = pd.read_csv(output_path, header=None)
print(check.head())

Train shape: (1322, 1559)
Test shape: (550, 1558)
Number of spectral features: 1555
Preprocessing complete: smoothing + derivative + scaling
Sample predictions: [203.46190113 167.3148899  155.88024289 147.35978337 142.26153704
 137.25080661 128.31341661 129.44824911 125.8171504  122.30568714]

Submission saved to: ../submissions/exp_nir_article_style.csv
    0           1
0  95  203.461901
1  96  167.314890
2  97  155.880243
3  98  147.359783
4  99  142.261537
